<a href="https://colab.research.google.com/github/Smolry/Smart-HSRP-detection/blob/dev/Input.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install opencv-python


In [ ]:
import cv2

# 0 = default webcam; change to video path for file input
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Cannot open camera")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Can't receive frame (stream end?). Exiting...")
        break

    # Display the frame
    cv2.imshow('Raw Video Feed', frame)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release camera
cap.release()
cv2.destroyAllWindows()


Error: Cannot open camera
Error: Can't receive frame (stream end?). Exiting...


In [ ]:
import cv2

cap = cv2.VideoCapture(0)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_raw.mp4', fourcc, 25.0, (640, 480))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    out.write(frame)
    cv2.imshow('Recording...', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import numpy as np

# ---------- Preprocessing Function ----------
def preprocess_light(frame):
    """
    Lightweight preprocessing:
    - Resize for faster processing
    - Gamma correction (brightness/contrast boost)
    - Mild denoising
    """
    # Resize frame for consistency
    frame = cv2.resize(frame, (640, 480))

    # Gamma correction
    gamma = 1.3
    invGamma = 1.0 / gamma
    table = np.array([(i / 255.0) ** invGamma * 255 for i in range(256)]).astype("uint8")
    frame = cv2.LUT(frame, table)

    # Mild denoising (keeps sharpness)
    frame = cv2.fastNlMeansDenoisingColored(frame, None, 5, 5, 7, 21)

    return frame


# ---------- Initialize Camera ----------
cap = cv2.VideoCapture(0)  # 0 for default webcam; or replace with video path

if not cap.isOpened():
    print("Error: Cannot open camera")
    exit()

# ---------- Setup Video Writer ----------
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('preprocessed_output.mp4', fourcc, 25.0, (640, 480))

print("Press 'q' to quit...")

# ---------- Real-time Processing Loop ----------
while True:
    ret, frame = cap.read()
    if not ret:
        print("Error: Frame not received. Exiting...")
        break

    # Apply lightweight preprocessing
    processed_frame = preprocess_light(frame)

    # Write the preprocessed frame to video
    out.write(processed_frame)

    # Display the processed frame
    cv2.imshow('Preprocessed Live Feed', processed_frame)

    # Exit on 'q' key
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ---------- Cleanup ----------
cap.release()
out.release()
cv2.destroyAllWindows()
print("Video saved as 'preprocessed_output.mp4'")
